# Chapter 13 — Probability Distributions in Machine Learning
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Understand which distributions appear naturally in ML models
- Connect log-likelihood to model training
- Visualize how distribution choice affects model behavior

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.special import expit  # sigmoid
import seaborn as sns

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Distributions in ML

Every ML model is implicitly a **probabilistic model**.  When you choose:
- **MSE loss** → you assume Gaussian noise (linear regression)
- **Cross-entropy loss** → you assume Bernoulli/Categorical output
- **Poisson loss** → you assume count data

The distribution dictates the **likelihood function** the model maximises.

In [ ]:
# --- Gaussian assumption behind MSE ---
x = np.linspace(-4, 4, 300)
for sigma in [0.5, 1.0, 2.0]:
    plt.plot(x, stats.norm.pdf(x, 0, sigma), label=f"σ={sigma}")
plt.title("Gaussian Noise Assumptions (MSE regression)")
plt.xlabel("Residual"); plt.ylabel("Density")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Log-Likelihood Derivation

For n i.i.d. observations from $N(\mu, \sigma^2)$:

$$\mathcal{L}(\mu, \sigma | x_1,\ldots,x_n) = \prod_{i=1}^n \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x_i-\mu)^2}{2\sigma^2}}$$

Taking logs (log-likelihood):
$$\ell = -n\ln\sigma - \frac{n}{2}\ln(2\pi) - \frac{1}{2\sigma^2}\sum_{i=1}^n(x_i-\mu)^2$$

Maximising $\ell$ w.r.t. $\mu$ gives $\hat\mu = \bar x$, and minimising
$\sum(x_i-\mu)^2$ (MSE) gives the **same answer** — MSE *is* MLE under Gaussian.

In [ ]:
# Visualise: negative log-likelihood surface for mu and sigma
x_data = rng.normal(3, 1.5, 50)
mu_grid = np.linspace(1, 5, 60)
sig_grid = np.linspace(0.5, 3.5, 60)
MU, SIG = np.meshgrid(mu_grid, sig_grid)
NLL = np.array([[-stats.norm.logpdf(x_data, m, s).sum()
                  for m in mu_grid] for s in sig_grid])

plt.figure(figsize=(8, 5))
cp = plt.contourf(MU, SIG, NLL, levels=20, cmap="RdYlGn_r")
plt.colorbar(cp, label="Negative Log-Likelihood")
best = np.unravel_index(NLL.argmin(), NLL.shape)
plt.scatter(MU[best], SIG[best], c="red", s=100, zorder=5, label="Minimum NLL")
plt.xlabel("μ"); plt.ylabel("σ"); plt.title("NLL Surface — Gaussian")
plt.legend(); plt.tight_layout(); plt.show()
print(f"MLE μ={MU[best]:.2f} (true 3), MLE σ={SIG[best]:.2f} (true 1.5)")

## 3. Simulation — Distributions for Classification

For **binary classification**, outputs follow a **Bernoulli** distribution.
The **logistic function** maps any real number to [0,1]:
$$\sigma(z) = \frac{1}{1+e^{-z}} = P(y=1|x)$$

In [ ]:
# Logistic regression — decision boundary visualisation
np.random.seed(42)
n = 200
X = rng.normal(0, 1, (n, 2))
true_w = np.array([1.5, -1.0]); true_b = 0.5
z = X @ true_w + true_b
p = expit(z)
y = rng.binomial(1, p)

# Colour by class
plt.figure(figsize=(7, 5))
plt.scatter(X[y==0, 0], X[y==0, 1], alpha=0.5, label="Class 0")
plt.scatter(X[y==1, 0], X[y==1, 1], alpha=0.5, label="Class 1")
# Decision boundary: w.x + b = 0
x1 = np.linspace(-3, 3, 100)
x2 = -(true_w[0]*x1 + true_b) / true_w[1]
plt.plot(x1, x2, "k--", label="Decision boundary")
plt.title("Bernoulli Output — Binary Classification")
plt.legend(); plt.tight_layout(); plt.show()

# Cross-entropy loss
eps = 1e-9
bce = -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))
print(f"Cross-entropy loss: {bce:.4f}")

## 4. Visualisation — Distribution Zoo for ML

| Task | Distribution | Loss function |
|------|-------------|--------------|
| Regression | Gaussian | MSE |
| Binary classification | Bernoulli | Binary cross-entropy |
| Multi-class | Categorical | Categorical cross-entropy |
| Count prediction | Poisson | Poisson NLL |
| Positive continuous | Gamma / Log-Normal | Gamma/Log-Normal NLL |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
dists = [
    ("Gaussian (regression)", stats.norm(3, 1), np.linspace(-1, 7, 200), False),
    ("Bernoulli (binary clf)", None, [0, 1], True),
    ("Categorical (multi-class)", None, [0, 1, 2, 3], True),
    ("Poisson (count pred)", stats.poisson(4), range(15), True),
    ("Gamma (positive cont.)", stats.gamma(3, scale=1), np.linspace(0, 12, 200), False),
    ("Log-Normal", stats.lognorm(0.7, scale=np.exp(1)), np.linspace(0, 15, 200), False),
]
for ax, (title, dist, xs, is_discrete) in zip(axes.flat, dists):
    if title.startswith("Bernoulli"):
        ax.bar([0, 1], [0.35, 0.65], color=["steelblue", "salmon"])
    elif title.startswith("Categorical"):
        ax.bar([0,1,2,3], [0.1, 0.5, 0.3, 0.1], color="steelblue")
    elif is_discrete:
        ax.bar(xs, dist.pmf(list(xs)), color="steelblue")
    else:
        ax.plot(xs, dist.pdf(xs), lw=2)
        ax.fill_between(xs, dist.pdf(xs), alpha=0.3)
    ax.set_title(title, fontsize=10)
plt.suptitle("Distributions in Machine Learning", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Real Dataset Exercise — Choosing the Right Distribution

We fit different distributions to the same dataset and compare log-likelihoods.

In [ ]:
from scipy.stats import norm, expon, gamma, lognorm

# Generate a right-skewed dataset (like income / latency)
data = rng.gamma(2, scale=3, size=500)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
candidates = [
    ("Normal", norm, norm.fit(data)),
    ("Exponential", expon, expon.fit(data)),
    ("Gamma", gamma, gamma.fit(data)),
    ("Log-Normal", lognorm, lognorm.fit(data)),
]
x_range = np.linspace(data.min(), data.max(), 200)
for ax, (name, dist_cls, params) in zip(axes, candidates):
    ll = dist_cls.logpdf(data, *params).sum()
    ax.hist(data, bins=30, density=True, alpha=0.5, color="steelblue")
    ax.plot(x_range, dist_cls.pdf(x_range, *params), "r-", lw=2)
    ax.set_title(f"{name}\nLog-L={ll:.0f}")
plt.suptitle("Distribution Fitting Comparison", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Practice Problems

**P1.** A neural network outputs a sigmoid value of 0.82 for a sample whose true label is 1.
Compute the **binary cross-entropy** for this single sample.

**P2.** You observe 8, 3, 5, 12, 2 support tickets per hour.
Fit a Poisson distribution (MLE) and compute the probability of receiving 10+ tickets in one hour.

**P3.** Why does minimising MSE loss and maximising Gaussian log-likelihood give the **same** parameters?
Write a one-paragraph proof.

In [ ]:
# P1
p_hat = 0.82; y_true = 1
bce_single = -(y_true * np.log(p_hat) + (1 - y_true) * np.log(1 - p_hat))
print(f"P1 BCE = {bce_single:.4f}")

# P2
tickets = np.array([8, 3, 5, 12, 2])
lam_mle = tickets.mean()
prob_10plus = 1 - stats.poisson.cdf(9, lam_mle)
print(f"P2 λ_MLE={lam_mle:.1f}, P(X>=10)={prob_10plus:.4f}")